# 057 — Aprendizaje por refuerzo profundo

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución explicada

**Ejercicio 1.** target = −1 + 0.9·3 = **1.7**; error TD = 1.7 − 2.0 = **−0.3**;
Q ← 2.0 + 0.5·(−0.3) = **1.85**. Segunda vez: error = 1.7 − 1.85 = −0.15 →
Q = **1.775**. Cada repetición acerca Q al target a ritmo geométrico (factor 1−α).

**Ejercicio 2.** γ=0: G = **1** (solo el presente). γ=0.5: 1+0.5+0.25+0.125+0.0625 =
**1.9375** (horizonte ~2 pasos). γ=0.9: **4.0951** (horizonte ~10 pasos, 1/(1−γ)).
γ fija cuántos pasos hacia el futuro "pesan" en la decisión.

**Ejercicio 3.** La acción greedy es la 2 (Q=2.0): p = (1−ε) + ε/3 = 0.7 + 0.1 =
**0.8**. Las otras dos: ε/3 = **0.1** cada una.

**Ejercicio 4.** π = (0.5, 0.5). Δθ = α·G·∇log π(a₁) = 0.1·(−1)·(0.5, −0.5) =
(−0.05, +0.05) → θ = (−0.05, 0.05) → π = softmax = (**0.475, 0.525**). Sí: la acción
que precedió a un retorno negativo pierde masa de probabilidad — el mecanismo básico
que, a escala, entrena políticas complejas.


In [ ]:
result = run_lab("robotics", seed=57)
assert result["kind"] == "robotics"
assert result["evidence"]
show(result)


In [ ]:
# Verificación numérica
import math

# Ejercicio 1
q, alpha, gamma, r, max_next = 2.0, 0.5, 0.9, -1.0, 3.0
target = r + gamma * max_next
for _ in range(2):
    q = q + alpha * (target - q)
    print("Q →", q)
assert abs(q - 1.775) < 1e-9

# Ejercicio 2
for g in (0.0, 0.5, 0.9):
    G = sum(g ** k for k in range(5))
    print(f"γ={g}: G={G:.4f}")

# Ejercicio 3
eps, n = 0.3, 3
p_greedy = (1 - eps) + eps / n
print("p(greedy) =", p_greedy, "| p(otras) =", eps / n)

# Ejercicio 4
theta = [0.0, 0.0]
pi = [0.5, 0.5]
grad_log = [1 - pi[0], -pi[1]]
G, alpha_pg = -1.0, 0.1
theta = [t + alpha_pg * G * g for t, g in zip(theta, grad_log)]
e = [math.exp(t) for t in theta]
pi_new = [v / sum(e) for v in e]
print("θ =", theta, "| π =", [round(p, 4) for p in pi_new])
assert pi_new[0] < 0.5


## Reflexión

1. ¿Por qué el TD target usa la red congelada θ⁻ y qué patología aparece si se calcula con la misma red que se está actualizando?
2. ¿Qué hace exactamente el max en el objetivo de Q-learning que lo convierte en off-policy, y por qué eso habilita el replay buffer?
3. En RLHF se optimiza una recompensa aprendida de preferencias humanas con PPO: ¿qué riesgos de "reward hacking" de esta clase aplican directamente?
